# Comprehensive Guide to Convolutional Neural Networks (CNNs)

This notebook is an in-depth exploration of Convolutional Neural Networks (CNNs), focusing on:
1. **Image Preprocessing**
2. **CNN Model Architecture Variations**
3. **Activation Functions**
4. **Optimizers**
5. **Loss Functions**
6. **Evaluation Metrics**

It is designed to provide clean, modular, and well-commented code, with intuitive explanations of trade-offs and design decisions, making it highly suitable for learning and interview preparation.


In [ ]:
import os
import cv2
import PIL.Image as Image
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, regularizers

# Check TensorFlow version
print(f"TensorFlow Version: {tf.__version__}")


## 1. Image Preprocessing

Image preprocessing consists of steps taken to format images before they are fed into the CNN. Operations like filtering, resizing, and adjusting contrast improve the quality of data the model learns from.


In [ ]:
# Download a sample image for demonstration using tf.keras.utils.get_file
sample_url = 'https://storage.googleapis.com/download.tensorflow.org/example_images/YellowT-Shirt.jpg'
image_path = tf.keras.utils.get_file('sample_image.jpg', origin=sample_url)

# Load image via OpenCV (reads as BGR) and convert to RGB
img_bgr = cv2.imread(image_path)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

def display_images(img1, title1, img2, title2):
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(img1)
    axes[0].set_title(title1)
    axes[0].axis('off')
    if len(img2.shape) == 2:
        axes[1].imshow(img2, cmap='gray')
    else:
        axes[1].imshow(img2)
    axes[1].set_title(title2)
    axes[1].axis('off')
    plt.show()

# --- 1. Gaussian Blur ---
# What it does: Smoothes the image by averaging pixel values with their neighbors via a Gaussian distribution.
# Why use it: Reduces high-frequency noise and detail, which can help the model focus on structural features rather than noise.
# Effect: The image becomes blurrier.
# Parameter changes: Increasing kernel size (e.g., from (5,5) to (15,15)) makes the blur stronger.
img_blur = cv2.GaussianBlur(img_rgb, (15, 15), 0)
display_images(img_rgb, "Original", img_blur, "Gaussian Blur")

# --- 2. Sharpening ---
# What it does: Enhances edges by subtracting smoothed versions of the image from the original.
# Why use it: Makes features and contours more prominent, helping edge-detection filters in early CNN layers.
# Effect: Edges appear crisper.
# Parameter changes: Changing the center value (e.g., from 9 to 11) or increasing negative weights intensifies the sharpening.
kernel_sharpening = np.array([[-1,-1,-1], 
                              [-1, 9,-1],
                              [-1,-1,-1]])
img_sharpened = cv2.filter2D(img_rgb, -1, kernel_sharpening)
display_images(img_rgb, "Original", img_sharpened, "Sharpened")

# --- 3. Edge Detection (Canny) ---
# What it does: Identifies strong gradients (edges) in the image.
# Why use it: Can be used as a separate channel or primary preprocessing step to force the network to only look at shapes.
# Effect: Produces a binary-like image mapping out contours.
# Parameter changes: Lowering thresholds (e.g., from 100,200 to 50,100) will detect more subtle/weaker edges (more noise).
img_gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
img_edges = cv2.Canny(img_gray, 100, 200)
display_images(img_rgb, "Original", img_edges, "Canny Edge Detection")

# --- 4. Brightness and Contrast Adjustments ---
# What it does: Linear transformation (alpha * x + beta). Alpha controls contrast, Beta controls brightness.
# Why use it: Standardizes varied lighting conditions in datasets (data augmentation).
# Effect: Image becomes brighter/darker, colors pop more or wash out.
# Parameter changes: Higher alpha > 1 increases contrast. Higher beta > 0 increases brightness.
alpha = 1.5 # Contrast control (1.0-3.0)
beta = 50   # Brightness control (0-100)
img_bc = cv2.convertScaleAbs(img_rgb, alpha=alpha, beta=beta)
display_images(img_rgb, "Original", img_bc, "Brightness & Contrast Adjusted")


## Loading Dataset (Cats vs Dogs)

Here we load the datset the user provided (`cats_vs_dogs`) via TensorFlow datasets and apply resizing and normalization as our preprocessing.


In [ ]:
# Load dataset
ds_train, ds_info = tfds.load(
    "cats_vs_dogs",
    split=["train[:80%]"],
    as_supervised=True,
    with_info=True
)
ds_train = ds_train[0]

ds_test = tfds.load(
    "cats_vs_dogs",
    split=["train[80%:90%]"],
    as_supervised=True,
)
ds_test = ds_test[0]

# Define central preprocessing logic for tf.data
# Normalize pixel values to [0, 1] range which helps gradient descent converge faster.
def preprocess(img, label):
    img = tf.image.resize(img, (64, 64)) / 255.0
    return img, label

BATCH_SIZE = 32

train_dataset = ds_train.map(preprocess).shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_dataset = ds_test.map(preprocess).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Dataset Loaded and Preprocessed.")


## 2. CNN Model Architecture Variations & 3. Activation Functions

We define 4 distinct architectures, comparing standard setups to deep networks and networks heavily employing regularizations. We also integrate various activation functions to examine their application context.

### Activation Functions Explanations:
1. **ReLU (Rectified Linear Unit)**: `max(0, x)`. Used universally in hidden layers. Efficient and prevents vanishing gradient problem, but can suffer from "dead neurons" if inputs are strongly negative.
2. **Leaky ReLU**: `max(alpha * x, x)`. Used where ReLU might die off. It provides a small, non-zero gradient for negative values.
3. **Sigmoid**: Range `(0, 1)`. Perfect for binary classification logit outputs (like predicting Cat vs Dog). If used in hidden layers of deep networks, it causes vanishing gradients.
4. **Tanh**: Range `(-1, 1)`. Often used in recurrent generic architectures or image generation (GAN outputs). Zero-centered, which is nicer than sigmoid, but still suffers from vanishing gradients in deep feed-forward networks.
5. **Softmax**: Range `(0, 1)`, sum=1. Converts raw scores to probabilities. Strictly used in the output layer for multi-class classification.


In [ ]:
# --- Model 1: Basic Shallow Network (Standard ReLU and Sigmoid) ---
def build_shallow_model():
    model = models.Sequential([
        # Purpose: Extracts low-level features (edges, colors).
        # What if removed: The model won't capture spatial hierarchies well.
        layers.Conv2D(32, 3, activation="relu", input_shape=(64, 64, 3)),
        # Purpose: Downsamples feature maps, reducing parameters and preventing overfitting by enforcing translation invariance.
        # What if removed: Model gets too large (memory intensive) and overfits easily.
        layers.MaxPooling2D(),
        
        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        # Purpose: Outputs probability [0, 1]. Best for Binary Crossentropy.
        # What if replaced: If replaced with ReLU, output might be > 1, breaking probabilistic interpretation.
        layers.Dense(1, activation="sigmoid")
    ], name="Shallow_Model")
    return model

# --- Model 2: Deeper Network with Dropout and LeakyReLU ---
def build_deep_leaky_model():
    model = models.Sequential([
        layers.Conv2D(32, 3, input_shape=(64, 64, 3)),
        layers.LeakyReLU(alpha=0.1), # Activation: LeakyReLU to prevent dead neurons.
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3),
        layers.LeakyReLU(alpha=0.1),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3),
        layers.LeakyReLU(alpha=0.1),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(256),
        layers.LeakyReLU(alpha=0.1),
        
        # Purpose: Zeroes out percent of connections randomly to force the network to learn robust features.
        # What if removed: Network would likely memorize training data (overfit quickly).
        layers.Dropout(0.5),
        
        layers.Dense(1, activation="sigmoid")
    ], name="Deep_LeakyReLU_Dropout")
    return model

# --- Model 3: Architecture with BatchNormalization and Tanh ---
def build_batchnorm_tanh_model():
    model = models.Sequential([
        layers.Conv2D(32, 3, input_shape=(64, 64, 3)),
        # Purpose: Normalizes layer inputs. Accelerates training, provides minor regularization.
        # What if removed: Training becomes slower (needs smaller learning rate) and prone to dead gradients when using Tanh.
        layers.BatchNormalization(),
        # Activation: Tanh (-1, 1). We use it here to see its effect, though ReLU is generally preferred in CNNs.
        layers.Activation("tanh"),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3),
        layers.BatchNormalization(),
        layers.Activation("tanh"),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation="tanh"),
        layers.Dense(1, activation="sigmoid")
    ], name="BatchNorm_Tanh")
    return model

# --- Model 4: Architecture with L2 Regularization ---
def build_l2_reg_model():
    model = models.Sequential([
        # Purpose: L2 (Ridge) Regularization penalizes large weight formulations via `kernel_regularizer`.
        # What if removed: Without it, weights can explode and strictly adapt to outlier samples.
        layers.Conv2D(64, 3, activation="relu", kernel_regularizer=regularizers.l2(0.001), input_shape=(64, 64, 3)),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
        layers.Dense(1, activation="sigmoid")
    ], name="L2_Regularization")
    return model

models_dict = {
    "Shallow": build_shallow_model(),
    "Deep_Leaky": build_deep_leaky_model(),
    "BatchNorm_Tanh": build_batchnorm_tanh_model(),
    "L2_Reg": build_l2_reg_model()
}

for name, m in models_dict.items():
    print(f"--- {name} Summary ---")
    m.summary()
    print("\n")


## 4. Optimizers

Optimizers dictate how the model weights are updated over the gradients.

1. **SGD (Stochastic Gradient Descent)**:
   - **How it works**: Updates parameters directly proportional to the negative gradient. Often paired with Momentum to overcome local minima.
   - **Pros/Cons**: Very stable, theoretically generalizes best, but very slow to converge.
2. **Adam (Adaptive Moment Estimation)**:
   - **How it works**: Uses moving averages of both the gradients and the squared gradients (adapts learning rate per parameter).
   - **Pros/Cons**: Fast convergence, usually the best default choice. Can sometimes overfit faster than SGD.
3. **RMSprop**:
   - **How it works**: Similar to Adam but without the momentum of the first-order gradient. Divides learning rate by an exponentially decaying average of squared gradients.
   - **Pros/Cons**: Great for RNNs, performs well on highly non-stationary objectives.
4. **Adagrad**:
   - **How it works**: Adapts learning rate to the parameters. Infrequent parameters get larger updates.
   - **Pros/Cons**: Good for sparse data. Disadvantage is the learning rate always decreases, eventually stopping learning.
5. **AdamW**:
   - **How it works**: Adam with explicitly decoupled weight decay formulation.
   - **Pros/Cons**: Solves L2 regularization issues in standard Adam, achieving better generalization.


In [ ]:
# Define optimizers mapping
def get_optimizers():
    return {
        "SGD": tf.keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
        "Adam": tf.keras.optimizers.Adam(learning_rate=0.001),
        "RMSprop": tf.keras.optimizers.RMSprop(learning_rate=0.001),
        "Adagrad": tf.keras.optimizers.Adagrad(learning_rate=0.01),
        "AdamW": tf.keras.optimizers.AdamW(learning_rate=0.001, weight_decay=1e-4) # Requires modern TF versions
    }


## 5. Loss Functions

The loss function is the metric the model actively tries to minimize using the Optimizer.

1. **Binary Crossentropy** (`binary_crossentropy`):
   - **When to use**: Binary classification problems (like Cats vs Dogs). Requires `sigmoid` output layer.
   - **If used incorrectly**: Will yield mathematical errors or nonsensical gradients if applied to non-probabilistic outputs or generalized multi-class tasks natively.
2. **Categorical Crossentropy** (`categorical_crossentropy`):
   - **When to use**: Multi-class problems with ONE-HOT encoded labels. Requires `softmax` output layer.
3. **Sparse Categorical Crossentropy** (`sparse_categorical_crossentropy`):
   - **When to use**: Multi-class problems with INTEGER labels (0, 1, 2...). Requires `softmax` output layer.
4. **Mean Squared Error (MSE)** (`mse`):
   - **When to use**: Regression problems (predicting continuous values like price, age, bounding box coordinates).
   - **If used incorrectly**: Using MSE for classification drastically slows down learning and struggles with punishing confident incorrect predictions compared to crossentropy.
5. **Categorical Hinge Loss** (`categorical_hinge`):
   - **When to use**: "Maximum-margin" classification, often used in SVMs.
   - **If used incorrectly**: May not converge as smoothly as crossentropy for probability-based model outputs.


## 6. Evaluation Metrics

Metrics do NOT affect the model's weights during training (unlike Loss Functions). They are for human interpretability.

1. **Accuracy**: 
   - **Measures**: `(True Postives + True Negatives) / Total`
   - **When important**: Balanced datasets. Unreliable if classes are strictly imbalanced.
2. **Precision**: 
   - **Measures**: `True Positives / (True Positives + False Positives)`
   - **When important**: When False Positives are very costly (e.g., spam detection).
3. **Recall**: 
   - **Measures**: `True Positives / (True Positives + False Negatives)`
   - **When important**: When False Negatives are fatal (e.g., medical diagnosis).
4. **F1-Score**: 
   - **Measures**: Harmonic mean between Precision and Recall.
   - **When important**: When you need balance between Precision and Recall.
5. **AUC (Area Under the ROC Curve)**: 
   - **Measures**: Probability that a random positive example is ranked higher than a random negative example.
   - **When important**: In evaluating binary classifiers regardless of probability threshold.
6. **Top-K Accuracy** (usually K=k like K=5 in ImageNet):
   - **Measures**: If the correct class is within the top K predicted probabilities.
   - **When important**: Multi-class problems with huge number of overlapping categories.


In [ ]:
metrics_list = [
    tf.keras.metrics.BinaryAccuracy(name='accuracy'),
    tf.keras.metrics.Precision(name='precision'),
    tf.keras.metrics.Recall(name='recall'),
    tf.keras.metrics.AUC(name='auc')
    # Note: F1-score is typically computed manually via Precision/Recall (2 * (P*R)/(P+R)) 
]


## Experimentation: Training the Models

We will iterate over our crafted models, assigning each a distinct Optimizer. We will train them for 1-2 epochs due to time/compute constraints, enough to showcase the execution. Loss function fundamentally used here is `binary_crossentropy`. (Others are configured based on task, but cats vs dogs is strictly binary).


In [ ]:
import time

experiment_history = {}

# We combine model + specific optimizer to test variations
experiments = [
    {"name": "Shallow_Adam", "model": build_shallow_model(), "optimizer": "Adam"},
    {"name": "Deep_Leaky_SGD", "model": build_deep_leaky_model(), "optimizer": "SGD"},
    {"name": "BatchNorm_RMSprop", "model": build_batchnorm_tanh_model(), "optimizer": "RMSprop"},
    {"name": "L2_AdamW", "model": build_l2_reg_model(), "optimizer": "AdamW"}
]

# Take a smaller subset for quick demonstration
demo_train = train_dataset.take(50) # 50 batches * 32 images = 1600 images
demo_test = test_dataset.take(10)   # 10 batches * 32 images = 320 images

EPOCHS = 2

for exp in experiments:
    print(f"\n{'='*50}\nRunning Experiment: {exp['name']}\n{'='*50}")
    
    m = exp['model']
    opt_name = exp['optimizer']
    opt_instance = get_optimizers()[opt_name]
    
    # Compile
    m.compile(
        optimizer=opt_instance,
        loss='binary_crossentropy', # Appropriate here: strict binary classification
        metrics=[
            tf.keras.metrics.BinaryAccuracy(name='accuracy'),
            tf.keras.metrics.Precision(name='precision'),
            tf.keras.metrics.Recall(name='recall'),
            tf.keras.metrics.AUC(name='auc')
        ]
    )
    
    start_time = time.time()
    # Train
    history = m.fit(
        demo_train,
        validation_data=demo_test,
        epochs=EPOCHS,
        verbose=1
    )
    time_taken = time.time() - start_time
    
    # Save results
    experiment_history[exp['name']] = history.history
    print(f"Completed {exp['name']} in {time_taken:.2f} seconds.")



## Plotting Core Metric Results

Visualization of the trade-offs during our quick demonstration.


In [ ]:
plt.figure(figsize=(15, 10))

# 1. Plot Validation Accuracies
plt.subplot(2, 2, 1)
for name, hist in experiment_history.items():
    plt.plot(hist['val_accuracy'], marker='o', label=name)
plt.title('Validation Accuracy across Architectures')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 2. Plot Validation Loss
plt.subplot(2, 2, 2)
for name, hist in experiment_history.items():
    plt.plot(hist['val_loss'], marker='o', label=name)
plt.title('Validation Loss across Architectures')
plt.xlabel('Epoch')
plt.ylabel('Binary Crossentropy Loss')
plt.legend()
plt.grid(True)

# 3. Plot Validation AUC
plt.subplot(2, 2, 3)
for name, hist in experiment_history.items():
    plt.plot(hist['val_auc'], marker='o', label=name)
plt.title('Validation AUC across Architectures')
plt.xlabel('Epoch')
plt.ylabel('AUC')
plt.legend()
plt.grid(True)

# 4. Plot Validation Precision
plt.subplot(2, 2, 4)
for name, hist in experiment_history.items():
    plt.plot(hist['val_precision'], marker='o', label=name)
plt.title('Validation Precision across Architectures')
plt.xlabel('Epoch')
plt.ylabel('Precision')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# Additional F1-Score custom calculation based on final epoch val metrics
print("\n--- Final Epoch Metrics Summary ---")
for name, hist in experiment_history.items():
    precision = hist['val_precision'][-1]
    recall = hist['val_recall'][-1]
    f1 = 2 * (precision * recall) / (precision + recall + 1e-7) # Add epsilon to avoid division by zero
    print(f"{name}:")
    print(f"  Accuracy : {hist['val_accuracy'][-1]:.4f}")
    print(f"  AUC      : {hist['val_auc'][-1]:.4f}")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall   : {recall:.4f}")
    print(f"  F1-Score : {f1:.4f}\n")


## Summary and Trade-Offs Highlight

1. **Shallow vs Deep Networks**: Shallow networks (`Shallow_Adam`) learn quickly and use very little memory, but fail to discover generalized, complex relationships in data like shapes and textures. Deep networks (`Deep_Leaky_SGD`) achieve deeper spatial understanding, but demand more compute power, time, and are susceptible to overfitting, necessitating Regularization techniques.
2. **Regularization (`L2_Reg`, `Dropout`)**: Added complex models heavily slow down overfitting (where validation loss starts rising but training loss falls). This is the tradeoff: regularization usually extends the number of epochs to converge to the optimal accuracy but keeps valid inference strong.
3. **Loss Choices**: Using the correct loss function strictly defines the gradient's vector space. Without `binary_crossentropy` on `cats_vs_dogs`, the gradients may bounce indefinitely causing failure to learn even on good architectures.
4. **Optimizers**: Adam converges extremely fast but its learning rate adaptivity can cause it to miss out on the final granular generalizations that pure `SGD` with momentum could squeeze out.
5. **Metrics Context**: High accuracy might be misleading if the dataset had 90% cats and 10% dogs (a model predicting purely 'cat' has 90% acc). This is why we examine `AUC` (Area Under ROC curve) which maps recall and false positive rates. Precision evaluates the confidence when it guesses Dog, and Recall evaluates how many of the actual dogs it managed to catch.
